# Notebook 16 — Robust Semantic Boundary Leverage and causal score validation

**Purpose.** Compute SABER's directed, alert-costed channel importance and test
whether it predicts the actual semantic harm caused by removing individual
channels better than magnitude, Taylor, Fisher, and random rankings.

This is the **G1 go/no-go experiment**. Do not begin the full SABER training
stack if R-SBL does not win on the prespecified harm metrics.

**Primary harm endpoints**

1. Action-Weighted Boundary Inversion Rate (AWBIR)
2. Hierarchical Semantic Risk (HSR)
3. Fine-type macro-F1 loss
4. Family macro-F1 loss

**Outputs**

- cost-profile-specific SBL tables and robust R-SBL
- resumable single-channel ablation table
- rank-correlation and top-k harm retrieval report
- `G1_score_gate.json`

In [ ]:
# Colab/repository bootstrap
from pathlib import Path
import os, sys, json, subprocess, platform

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# Override with %env SABER_REPO=/your/path if your repository is elsewhere.
candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. Set SABER_REPO or edit the candidate path."
    )
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("Repository:", REPO)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())

In [ ]:
# Install only the small SABER extension requirements.
# The original repository requirements must already be installed.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-saber.txt"],
        check=True,
    )

In [ ]:
import yaml
from src.saber.adapters import SaberRepo

repo = SaberRepo.discover(REPO)
with open(REPO / "config" / "saber.yaml", "r", encoding="utf-8") as handle:
    SABER_CFG = yaml.safe_load(handle)

OUTPUT_ROOT = repo.output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("SABER output root:", OUTPUT_ROOT)
print("Config branch:", SABER_CFG["project"]["branch"])

In [ ]:
# Repository bridge: auto-discovery first, explicit override second.
#
# If auto-discovery fails, set these objects using the same loader/model
# construction cells from the completed Computer Networks notebooks:
#   TRAIN_LOADER = ...
#   VAL_LOADER = ...
#   TEST_LOADER = ...
#   MODEL = ...
#   CLASS_NAMES = [...]
#
# MODEL must be the uncompressed CNN1D anchor and loaders must use the frozen
# train/validation/test split.

import torch
from src.saber.adapters import (
    auto_discover_loaders,
    auto_build_cnn,
    discover_anchor_checkpoint,
    infer_class_names_from_results,
    unpack_batch,
)

TRAIN_LOADER = globals().get("TRAIN_LOADER")
VAL_LOADER = globals().get("VAL_LOADER")
TEST_LOADER = globals().get("TEST_LOADER")
MODEL = globals().get("MODEL")
CLASS_NAMES = globals().get("CLASS_NAMES")

if any(obj is None for obj in (TRAIN_LOADER, VAL_LOADER, TEST_LOADER)):
    TRAIN_LOADER, VAL_LOADER, TEST_LOADER, _DATA_BUNDLE = auto_discover_loaders(repo)

first_batch = next(iter(VAL_LOADER))
x0, y0, env0 = unpack_batch(first_batch)
raw_example = x0[: min(8, len(x0))].float()

if CLASS_NAMES is None:
    CLASS_NAMES = infer_class_names_from_results(repo)

if MODEL is None:
    checkpoint = discover_anchor_checkpoint(repo)
    MODEL, MODEL_FACTORY_ERRORS = auto_build_cnn(
        n_features=int(x0.shape[-1]),
        n_classes=len(CLASS_NAMES),
        checkpoint=checkpoint,
    )
    print("Loaded checkpoint:", checkpoint)
    if MODEL_FACTORY_ERRORS:
        print("Model factory attempts that were skipped:", MODEL_FACTORY_ERRORS)

# Infer whether the historical CNN expects [B,F] and unsqueezes internally or
# expects an explicit [B,1,F] tensor. This decision is frozen for the notebook.
MODEL_INPUT_MODE = None
probe_out = None
candidate_inputs = [("raw", raw_example)]
if raw_example.ndim == 2:
    candidate_inputs.append(("unsqueeze_channel", raw_example.unsqueeze(1)))
errors = {}
MODEL.eval()
for mode, candidate in candidate_inputs:
    try:
        with torch.no_grad():
            probe_out = MODEL(candidate)
        MODEL_INPUT_MODE = mode
        EXAMPLE_INPUT = candidate
        break
    except Exception as exc:
        errors[mode] = repr(exc)

if MODEL_INPUT_MODE is None:
    raise RuntimeError(
        "Could not infer the CNN input convention. Set EXAMPLE_INPUT and "
        "MODEL_INPUT manually. Attempts: " + json.dumps(errors, indent=2)
    )

def MODEL_INPUT(x):
    if MODEL_INPUT_MODE == "unsqueeze_channel" and x.ndim == 2:
        return x.unsqueeze(1)
    return x

if probe_out.shape[1] != len(CLASS_NAMES):
    raise RuntimeError(
        f"Model outputs {probe_out.shape[1]} classes but CLASS_NAMES has "
        f"{len(CLASS_NAMES)} entries. Supply the exact label-encoder order."
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL = MODEL.to(DEVICE)
EXAMPLE_INPUT = EXAMPLE_INPUT.to(DEVICE)
print("Device:", DEVICE)
print("Model:", type(MODEL).__name__)
print("Classes:", len(CLASS_NAMES))
print("Input convention:", MODEL_INPUT_MODE, tuple(EXAMPLE_INPUT.shape))

## 1. Load the frozen graph and baseline group scores

In [ ]:
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.risk_graph import aggregate_robust_edge_weights
from src.saber.leverage import (
    semantic_boundary_leverage,
    robust_profile_aggregation,
    merge_score_tables,
    validate_score_against_harm,
)

GRAPH_DIR = OUTPUT_ROOT / "14_risk_graph"
BASELINE_DIR = OUTPUT_ROOT / "15_baselines"
OUT = OUTPUT_ROOT / "16_score_validation"
OUT.mkdir(parents=True, exist_ok=True)

taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
groups = pd.read_csv(BASELINE_DIR / "prunable_groups.csv")
baseline_scores = pd.read_csv(BASELINE_DIR / "group_baseline_scores.csv")
edges_by_profile = pd.read_csv(GRAPH_DIR / "asvg_edges_by_cost_profile.csv")
robust_graph = pd.read_csv(GRAPH_DIR / "asvg_edges_robust.csv")

assert set(groups["group_id"]) == set(baseline_scores["group_id"])
assert np.isclose(robust_graph["robust_weight"].sum(), 1.0)
print("Groups:", len(groups), "| Robust edges:", len(robust_graph))

## 2. Compute SBL under each cost profile and robustly aggregate

A backward pass is made per directed edge on a capped, class-balanced validation
subset. The cap is a compute control; repeat later at multiple caps to document
score stability before confirmatory experiments.

In [ ]:
profile_score_tables = {}
edge_contribution_tables = []

for profile_name in DEFAULT_COST_PROFILES:
    profile_edges = edges_by_profile[
        edges_by_profile["cost_profile"] == profile_name
    ].copy()
    table, edge_table = semantic_boundary_leverage(
        MODEL,
        VAL_LOADER,
        groups,
        profile_edges,
        device=DEVICE,
        max_samples_per_class=int(
            SABER_CFG["score_validation"]["max_samples_per_source_class"]
        ),
        edge_weight_column="normalized_weight",
        normalize_by_group_size=float(
            SABER_CFG["score_validation"]["group_size_alpha"]
        ),
        normalize_by_flops=float(SABER_CFG["score_validation"]["flops_beta"]),
        input_transform=MODEL_INPUT,
    )
    table["cost_profile"] = profile_name
    edge_table["cost_profile"] = profile_name
    profile_score_tables[profile_name] = table
    edge_contribution_tables.append(edge_table)
    table.to_csv(OUT / f"sbl_scores_{profile_name}.csv", index=False)

robust_scores = robust_profile_aggregation(
    profile_score_tables,
    score_column="sbl",
    method="cvar",
    q=float(SABER_CFG["risk_graph"]["robust_cvar_q"]),
)
all_scores = merge_score_tables(baseline_scores, robust_scores)
all_scores.to_csv(OUT / "group_scores_with_r_sbl.csv", index=False)
pd.concat(edge_contribution_tables, ignore_index=True).to_csv(
    OUT / "edge_group_leverage_long.csv", index=False
)
display(all_scores.nlargest(20, "r_sbl")[
    ["group_id", "module_path", "channel_index", "magnitude", "taylor", "fisher", "r_sbl"]
])

## 3. Build a capped, balanced validation ablation set

Single-channel ablation must be evaluated on the same records for every group.
The subset is sampled deterministically by class and stored as tensors for
resumption.

In [ ]:
from src.saber.adapters import unpack_batch

ABLATION_CACHE = OUT / "balanced_ablation_subset.pt"
MAX_PER_CLASS = 512

if ABLATION_CACHE.exists():
    bundle = torch.load(ABLATION_CACHE, map_location="cpu")
    X_ABL = bundle["x"]
    Y_ABL = bundle["y"].long()
else:
    buckets = {c: [] for c in range(taxonomy.n_classes)}
    counts = {c: 0 for c in range(taxonomy.n_classes)}
    for batch in VAL_LOADER:
        x, y, _ = unpack_batch(batch)
        y_np = y.detach().cpu().numpy()
        for c in range(taxonomy.n_classes):
            need = MAX_PER_CLASS - counts[c]
            if need <= 0:
                continue
            idx = np.flatnonzero(y_np == c)[:need]
            if len(idx):
                buckets[c].append(x[idx].detach().cpu())
                counts[c] += len(idx)
        if all(v >= MAX_PER_CLASS for v in counts.values()):
            break
    xs, ys = [], []
    for c, chunks in buckets.items():
        if not chunks:
            continue
        xc = torch.cat(chunks, dim=0)[:MAX_PER_CLASS]
        xs.append(xc)
        ys.append(torch.full((len(xc),), c, dtype=torch.long))
    X_ABL = torch.cat(xs, dim=0)
    Y_ABL = torch.cat(ys, dim=0)
    torch.save({"x": X_ABL, "y": Y_ABL}, ABLATION_CACHE)

print("Ablation subset:", tuple(X_ABL.shape))
print(pd.Series(Y_ABL.numpy()).value_counts().sort_index().describe())

## 4. Cache teacher outputs and reference metrics

In [ ]:
from src.saber.metrics import (
    action_weighted_boundary_inversion_rate,
    full_model_audit,
    hierarchical_semantic_risk,
    softmax_np,
)
from src.saber.taxonomy import DEFAULT_COST_PROFILES

MODEL.eval()
teacher_chunks = []
batch_size = 4096
with torch.no_grad():
    for start in range(0, len(X_ABL), batch_size):
        teacher_chunks.append(
            MODEL(MODEL_INPUT(X_ABL[start:start+batch_size].to(DEVICE))).cpu()
        )
TEACHER_LOGITS = torch.cat(teacher_chunks).numpy()
TEACHER_PRED = TEACHER_LOGITS.argmax(1)
teacher_audit = full_model_audit(
    TEACHER_LOGITS, Y_ABL.numpy(), taxonomy, DEFAULT_COST_PROFILES
)
pd.DataFrame([teacher_audit]).to_csv(OUT / "teacher_ablation_subset_audit.csv", index=False)
teacher_audit

## 5. Single-group causal ablation

The loop is resumable. It functionally removes each channel and its immediate
downstream dependency, then measures semantic harm with no fine-tuning. This is
the causal target the pruning score is intended to predict.

In [ ]:
from sklearn.metrics import f1_score
from src.saber.surgery import temporarily_zero_group
from src.saber.metrics import (
    action_weighted_boundary_inversion_rate,
    full_model_audit,
)

ABLATION_CSV = OUT / "single_group_causal_ablation.csv"
if ABLATION_CSV.exists():
    existing = pd.read_csv(ABLATION_CSV)
    completed = set(existing["group_id"])
    records = existing.to_dict(orient="records")
    print("Resuming with", len(completed), "completed groups.")
else:
    completed = set()
    records = []

for group_idx, group in enumerate(groups.to_dict(orient="records"), start=1):
    gid = str(group["group_id"])
    if gid in completed:
        continue

    student_chunks = []
    with temporarily_zero_group(MODEL, group):
        MODEL.eval()
        with torch.no_grad():
            for start in range(0, len(X_ABL), batch_size):
                student_chunks.append(
                    MODEL(MODEL_INPUT(X_ABL[start:start+batch_size].to(DEVICE))).cpu()
                )
    student_logits = torch.cat(student_chunks).numpy()
    student_audit = full_model_audit(
        student_logits, Y_ABL.numpy(), taxonomy, DEFAULT_COST_PROFILES
    )
    awbir, _ = action_weighted_boundary_inversion_rate(
        TEACHER_LOGITS,
        student_logits,
        Y_ABL.numpy(),
        robust_graph,
        weight_column="robust_weight",
    )

    record = {
        "group_id": gid,
        "module_path": group["module_path"],
        "channel_index": int(group["channel_index"]),
        "parameter_cost": int(group["parameter_cost"]),
        "flops_cost": float(group["flops_cost"]),
        "harm_awbir": float(awbir),
        "harm_hsr_balanced_soc": float(
            student_audit["hsr_balanced_soc"] - teacher_audit["hsr_balanced_soc"]
        ),
        "harm_fine_macro_f1": float(
            teacher_audit["fine_macro_f1"] - student_audit["fine_macro_f1"]
        ),
        "harm_family_macro_f1": float(
            teacher_audit["family_macro_f1"] - student_audit["family_macro_f1"]
        ),
        "harm_benign_false_alert": float(
            student_audit["benign_to_attack_rate"]
            - teacher_audit["benign_to_attack_rate"]
        ),
        "student_fine_macro_f1": float(student_audit["fine_macro_f1"]),
        "student_family_macro_f1": float(student_audit["family_macro_f1"]),
    }
    records.append(record)

    if group_idx % 10 == 0 or group_idx == len(groups):
        pd.DataFrame(records).to_csv(ABLATION_CSV, index=False)
        print(f"Saved {len(records)}/{len(groups)} groups")

ablation = pd.DataFrame(records)
assert len(ablation) == len(groups)
display(ablation.nlargest(15, "harm_awbir"))

## 6. Score–harm validation and G1 decision

The preregistered interpretation is directional:

- scores are large for groups that should be retained;
- harm is large when deleting a group is damaging;
- therefore positive rank correlation is desirable.

R-SBL should beat magnitude and Taylor on at least two primary harm endpoints.

In [ ]:
analysis = all_scores.merge(ablation, on=[
    "group_id", "module_path", "channel_index", "parameter_cost", "flops_cost"
], how="inner")

score_columns = ["random", "magnitude", "taylor", "fisher", "bn_gamma", "r_sbl"]
harm_columns = [
    "harm_awbir",
    "harm_hsr_balanced_soc",
    "harm_fine_macro_f1",
    "harm_family_macro_f1",
]
validation = validate_score_against_harm(
    analysis,
    score_columns=score_columns,
    harm_columns=harm_columns,
    top_k_fraction=float(
        SABER_CFG["score_validation"]["top_k_harm_fraction"]
    ),
)
validation.to_csv(OUT / "score_harm_validation.csv", index=False)
analysis.to_csv(OUT / "score_and_causal_harm.csv", index=False)
display(validation.sort_values(["harm", "spearman"], ascending=[True, False]))

wins = 0
win_details = {}
for harm in harm_columns:
    frame = validation[validation["harm"] == harm].set_index("score")
    if "r_sbl" not in frame.index:
        continue
    r_value = float(frame.loc["r_sbl", "spearman"])
    comparator = max(
        float(frame.loc[s, "spearman"])
        for s in ("magnitude", "taylor")
        if s in frame.index
    )
    advantage = r_value - comparator
    won = advantage >= float(
        SABER_CFG["score_validation"]["required_spearman_advantage"]
    )
    wins += int(won)
    win_details[harm] = {
        "r_sbl_spearman": r_value,
        "best_magnitude_taylor": comparator,
        "advantage": advantage,
        "won": won,
    }

gate_pass = wins >= int(
    SABER_CFG["gates"]["G1_score"]["minimum_primary_metrics_won"]
)
gate = {
    "gate": "G1_score_validity",
    "passed": bool(gate_pass),
    "primary_metrics_won": int(wins),
    "required": int(SABER_CFG["gates"]["G1_score"]["minimum_primary_metrics_won"]),
    "details": win_details,
}
with open(OUT / "G1_score_gate.json", "w", encoding="utf-8") as handle:
    json.dump(gate, handle, indent=2)
gate

## 7. Diagnostic figures

These plots should become the first result figure only if G1 passes.

In [ ]:
import matplotlib.pyplot as plt

for harm in harm_columns:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, score in zip(axes, ["magnitude", "taylor", "r_sbl"]):
        ax.scatter(analysis[score], analysis[harm], alpha=0.7)
        rho = validation[
            (validation["score"] == score) & (validation["harm"] == harm)
        ]["spearman"].iloc[0]
        ax.set_title(f"{score}: Spearman={rho:.2f}")
        ax.set_xlabel(score)
        ax.set_ylabel(harm)
    fig.tight_layout()
    safe_name = harm.replace("harm_", "")
    fig.savefig(OUT / f"score_vs_{safe_name}.png", dpi=250)
    plt.show()

## 8. Interpretation

- **Pass:** proceed to Notebook 17 and preserve this exact graph/score config.
- **Borderline:** repeat the score at 256/1,024 samples per class and inspect
  environment/cost aggregation before redesigning.
- **Fail:** do not hide the result. Revisit the edge weighting or test a
  second-order estimate before investing in full SABER training.

In [ ]:
import subprocess
from src.saber.adapters import save_run_manifest

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    git_commit = None

artefacts = [
    OUT / "group_scores_with_r_sbl.csv",
    OUT / "edge_group_leverage_long.csv",
    OUT / "single_group_causal_ablation.csv",
    OUT / "score_harm_validation.csv",
    OUT / "score_and_causal_harm.csv",
    OUT / "G1_score_gate.json",
]
save_run_manifest(
    OUT / "manifest.json",
    notebook="16_saber_boundary_leverage.ipynb",
    config=SABER_CFG,
    artifacts=artefacts,
    git_commit=git_commit,
)
print("G1 PASS" if gate_pass else "G1 DID NOT PASS")